In this notebook I will be getting any new games that have occurred. I will check with the current values in my pbp database, and add any games not located in it.

In [3]:
from ryp import r, to_py
import pandas as pd
import numpy as np

In [4]:
# Setup R environment and install packages
r('''
    # 1. Set CRAN mirror
    options(repos = c(CRAN = "https://cloud.r-project.org/"))
    
    # 2. Define packages
    packages <- c("pak", "tictoc", "svglite", "arrow")
    
    # 3. Install missing packages
    for (pkg in packages) {
        if (!require(pkg, character.only = TRUE)) {
            print(paste("Installing", pkg, "..."))
            install.packages(pkg, dependencies = TRUE)
        } else {
            print(paste(pkg, "is already installed"))
        }
    }
    
    # 4. Load all packages
    lapply(packages, library, character.only = TRUE)
    print("All packages ready!")
''')

# Now your Python code can use the R packages
print("R packages are ready for use!")

[1] "pak is already installed"
[1] "tictoc is already installed"
[1] "svglite is already installed"
[1] "arrow is already installed"
[[1]]
 [1] "tictoc"    "pak"       "arrow"     "svglite"   "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[2]]
 [1] "tictoc"    "pak"       "arrow"     "svglite"   "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[3]]
 [1] "tictoc"    "pak"       "arrow"     "svglite"   "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "tictoc"    "pak"       "arrow"     "svglite"   "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[1] "All packages ready!"
R packages are ready for use!


Loading required package: pak
Loading required package: tictoc


In [5]:
#Must use pak to load wehoop from GitHub. Version 2.1.0 is on CRAN, but it does not contain the functions to load the roster and team data
r('''
    pak::pak("sportsdataverse/wehoop")
''')

ℹ Loading metadata database
✔ Loading metadata database ... done

 
→ Package library at '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/library'.
ℹ No downloads are needed
✔ 1 pkg + 43 deps: kept 17 [6.1s]


In [6]:
r('''
    tictoc::tic()
    wnba_pbp <- wehoop::load_wnba_pbp()
    tictoc::toc()
''')



1.505 sec elapsed


In [11]:
#Converting the Extracted data to a pandas dataframe, from here I will check if there's any new game_id's so I can add any new games to the database
new_wnba_pbp = to_py('wnba_pbp', format = 'pandas') 
old_wnba_pbp = pd.read_csv("data/wnba_raw.csv")

#Getting all of the new games that aren't in the current database
old_game_ids = old_wnba_pbp["game_id"].unique()
new_game_ids = new_wnba_pbp["game_id"].unique()

game_id_mask = np.isin(new_game_ids, old_game_ids)
new_games = new_game_ids[~game_id_mask]

#Getting these rows from new_wnba_pbp
new_games = new_wnba_pbp[new_wnba_pbp["game_id"].isin(new_games)]
new_games

,        20T17:04,:3,4Z2026-06-20T17,:04:54Z,2026-06-2,0T17,:04:55Z202,6-06-20T17,:05:00Z2026-0,6-20T17:05:08Z2026-0,...,6-20T17:05:21Z2026-06-20T1,7:05:36Z2026-06-20T17:05:5,0Z2026,-06-20T,17:06:12,Z2026-06-20T,17:06:27Z202,6-06-20T1,7:06:48Z2026-0,6-20T17:07:18Z3:0
index,,,,,,,,,,,,,,,,,,,,,
1,1,4.018570e+09,4,615,        ,Natasha Howard vs. Awak Kuier,0,0,1,1st Quarter,...,1200.0,2400.0,1,NaN,NaN,-2.147484e+08,-214748365.0,2026-06-28,2026-06-28 14:00:00-04:00,NaN
2,2,4.018570e+09,8,128,                         ,Olivia Miles misses two point shot,0,0,1,1st Quarter,...,1180.0,2380.0,1,1.0,1.0,-3.775000e+01,2.0,2026-06-28,2026-06-28 14:00:00-04:00,NaN
3,3,4.018570e+09,9,155,                 ,Jessica Shepard defensive rebound,0,0,1,1st Quarter,...,1175.0,2375.0,1,1.0,1.0,3.775000e+01,-2.0,2026-06-28,2026-06-28 14:00:00-04:00,NaN
4,4,4.018570e+10,10,119,                 ,Awak Kuier makes 2-foot two point shot (Jessic...,0,2,1,1st Quarter,...,1153.0,2353.0,1,1.0,1.0,3.975000e+01,-0.0,2026-06-28,2026-06-28 14:00:00-04:00,NaN
5,5,4.018570e+10,14,132,                   ,Nia Coffey makes 23-foot three point step back...,3,2,1,1st Quarter,...,1135.0,2335.0,1,1.0,1.0,-3.275000e+01,21.0,2026-06-28,2026-06-28 14:00:00-04:00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10466,396,4.018570e+11,591,99,Free Throw - 2 of 2,Breanna Stewart makes free throw 2 of 2,97,95,4,4th Quarter,...,0.2,0.2,4,4.0,2.0,-2.800000e+01,0.0,2026-06-21,2026-06-21 20:00:00-04:00,NaN
10467,397,4.018570e+11,592,92,Jump Shot,Nneka Ogwumike makes 25-foot three point jumpe...,97,98,4,4th Quarter,...,0.0,0.0,4,4.0,2.0,2.375000e+01,18.0,2026-06-21,2026-06-21 20:00:00-04:00,NaN
10468,398,4.018570e+11,594,280,Ref-Initiated Review (Stands),(00:00) [Liberty] REF-INITIATED REVIEW (CALL S...,97,98,4,4th Quarter,...,0.0,0.0,4,4.0,2.0,-2.147484e+08,-214748365.0,2026-06-21,2026-06-21 20:00:00-04:00,NaN


In [12]:
#Appending these values to the end of the database
new_games.to_csv("data/wnba_raw.csv", mode = "a", header = False, index = False)